In [21]:
import pandas as pd
import numpy as np

In [22]:
# we first load the metadata from MIMIC-CXR-JPG
meta = pd.read_csv('metadata/mimic-cxr-2.0.0-metadata.csv.gz')
chexpert = pd.read_csv('metadata/mimic-cxr-2.0.0-chexpert.csv.gz')
split = pd.read_csv('metadata/mimic-cxr-2.0.0-split.csv.gz')

In [23]:
# we focus on frontal views only (AP or PA)
frontal = meta[meta['ViewPosition'].isin(['PA', 'AP'])].copy()
print(f'Frontal images available: {len(frontal)}')

Frontal images available: 243334


In [24]:
# merging with labels and split (meta with chexpert)
df = frontal.merge(chexpert, on = ['subject_id', 'study_id'], how = 'inner')
df = df.merge(split, on = ['subject_id', 'study_id', 'dicom_id'], how = 'inner')

In [25]:
df

,dicom_id,subject_id,study_id,PerformedProcedureStepDescription,ViewPosition,Rows,Columns,StudyDate,StudyTime,ProcedureCodeSequence_CodeMeaning,...,Fracture,Lung Lesion,Lung Opacity,No Finding,Pleural Effusion,Pleural Other,Pneumonia,Pneumothorax,Support Devices,split
0,02aa804e-bde0afdd-112c0b34-7bc16630-4e384014,10000032,50414267,CHEST (PA AND LAT),PA,3056,2544,21800506,213014.531,CHEST (PA AND LAT),...,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,train
1,2a2277a9-b0ded155-c0de8eb9-c124d10e-82c5caab,10000032,53189527,CHEST (PA AND LAT),PA,3056,2544,21800626,165500.312,CHEST (PA AND LAT),...,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,train
2,68b5c4b1-227d0485-9cc38c3f-7b84ab51-4b472714,10000032,53911762,CHEST (PORTABLE AP),AP,2705,2539,21800723,80556.875,CHEST (PORTABLE AP),...,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,train
3,fffabebf-74fd3a1f-673b6b41-96ec0ac9-2ab69818,10000032,53911762,CHEST (PORTABLE AP),AP,2906,2258,21800723,80556.875,CHEST (PORTABLE AP),...,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,train
4,ea030e7a-2e3b1346-bc518786-7a8fd698-f673b44c,10000032,56699142,CHEST (PORTABLE AP),AP,3056,2544,21800805,234424.765,CHEST (PORTABLE AP),...,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,train
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
243319,3fcd0406-9b111603-feae7033-96632b3a-111333e5,19999733,57132437,CHEST (PA AND LAT),PA,3056,2544,21520708,224550.171,CHEST (PA AND LAT),...,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,train
243320,428e2c18-5721d8f3-35a05001-36f3d080-9053b83c,19999733,57132437,CHEST (PA AND LAT),PA,3056,2544,21520708,224550.171,CHEST (PA AND LAT),...,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,train
243321,58766883-376a15ce-3b323a28-6af950a0-16b793bd,19999987,55368167,CHEST (PORTABLE AP),AP,2544,3056,21451104,51448.218,CHEST (PORTABLE AP),...,NaN,0.0,NaN,NaN,0.0,NaN,NaN,0.0,NaN,train
243322,7ba273af-3d290f8d-e28d0ab4-484b7a86-7fc12b08,19999987,58621812,CHEST (PORTABLE AP),AP,3056,2544,21451102,202809.234,CHEST (PORTABLE AP),...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,train


In [26]:
# storing the 14 pathology columns
LABELS = ['Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema',
         'Enlarged Cardiomediastinum', 'Fracture', 'Lung Lesion',
         'Lung Opacity', 'No Finding', 'Pleural Effusion', 'Pleural Other',
         'Pneumonia', 'Pneumothorax', 'Support Devices']

In [27]:
# fill NaN with 0 and replace -1 (uncertain) as 0.
# NaN shows that the condition wasn't recorded, -1 represents uncertainty about the said condition.
# NaN/uncertain recordings will be treated as negative for simplicity now.
df[LABELS] = df[LABELS].fillna(0).replace(-1, 0)

In [29]:
df[LABELS]

,Atelectasis,Cardiomegaly,Consolidation,Edema,Enlarged Cardiomediastinum,Fracture,Lung Lesion,Lung Opacity,No Finding,Pleural Effusion,Pleural Other,Pneumonia,Pneumothorax,Support Devices
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
243319,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
243320,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
243321,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
243322,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


In [34]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 243324 entries, 0 to 243323
Data columns (total 27 columns):
 #   Column                                      Non-Null Count   Dtype  
---  ------                                      --------------   -----  
 0   dicom_id                                    243324 non-null  str    
 1   subject_id                                  243324 non-null  int64  
 2   study_id                                    243324 non-null  int64  
 3   PerformedProcedureStepDescription           225259 non-null  str    
 4   ViewPosition                                243324 non-null  str    
 5   Rows                                        243324 non-null  int64  
 6   Columns                                     243324 non-null  int64  
 7   StudyDate                                   243324 non-null  int64  
 8   StudyTime                                   243324 non-null  float64
 9   ProcedureCodeSequence_CodeMeaning           243324 non-null  str    
 10  ViewCod

In [53]:
# stratified sample of ~5000 studies with oversampling of rare pathologies
TARGET_TOTAL = 5000
train_pool = df[df['split'] == 'train']

sampled_ids = set()
for label in LABELS:
    positive = train_pool[train_pool[label] == 1]
    take = positive.sample(n = min(400, len(positive)), random_state = 42)
    sampled_ids.update(take['dicom_id'].tolist())

In [54]:
# if under 4500 samples, we randomly fill the remaining samples
remaining = TARGET_TOTAL - len(sampled_ids)
if remaining > 0:
    leftover_pool = train_pool[~train_pool['dicom_id'].isin(sampled_ids)]
    extra = leftover_pool.sample(n = min(remaining, len(leftover_pool)), random_state = 42)
    sampled_ids.update(extra['dicom_id'].tolist())

In [55]:
subset = df[df['dicom_id'].isin(sampled_ids)]
print(f'Final subset size: {len(subset)}')
print(subset[LABELS].sum().sort_values(ascending = False))

Final subset size: 5537
Support Devices               2407.0
Pleural Effusion              2152.0
Lung Opacity                  1950.0
Atelectasis                   1698.0
Cardiomegaly                  1541.0
Edema                         1115.0
Pneumonia                      828.0
Consolidation                  758.0
Pneumothorax                   678.0
Enlarged Cardiomediastinum     603.0
Lung Lesion                    578.0
Fracture                       522.0
No Finding                     455.0
Pleural Other                  450.0
dtype: float64


In [56]:
# we'll hold out a small val/test slice from the splits in the data
val_subset = df[df['split'] == 'validate'].sample(n = 300, random_state = 42)
test_subset = df[df['split'] == 'test'].sample(n = 300, random_state = 42)

In [57]:
subset.to_csv('subset_train.csv', index = False)
val_subset.to_csv('subset_val.csv', index = False)
test_subset.to_csv('subset_test.csv', index = False)

-----------------------------------

In [58]:
# we now build the URL list directly from the subset IDs

In [60]:
train = pd.read_csv('subset_train.csv')
val = pd.read_csv('subset_val.csv')
test = pd.read_csv('subset_test.csv')

In [61]:
all_subset = pd.concat([train, val, test], ignore_index = True)

In [63]:
# we replicate the directory structure of MIMIC-CXR-JPG
def build_path(row):
    subj = f"p{str(row['subject_id'])[:2]}/p{row['subject_id']}"
    study = f"s{row['study_id']}"
    return f"files/{subj}/{study}/{row['dicom_id']}.jpg"

all_subset['jpg_path'] = all_subset.apply(build_path, axis = 1)
all_subset[['jpg_path']].to_csv('download_paths.csv', index = False)
print(f'Total images to be downloaded: {len(all_subset)}')

Total images to be downloaded: 6137


----------------------------------

In [67]:
# and finally, we download all images for the training set, the validation set and the testing set
import requests
import os
from requests.auth import HTTPBasicAuth
from tqdm import tqdm
import getpass

In [68]:
BASE_URL = "https://physionet.org/files/mimic-cxr-jpg/2.1.0/"
OUT_DIR = r"D:\omer files\projects\NMIMS\cxr_project\images"

username = 'omerresearches'
password = getpass.getpass('PhysioNet password: ')
auth = HTTPBasicAuth(username, password)

paths = pd.read_csv('download_paths.csv')['jpg_path'].tolist()

session = requests.Session()
session.auth = auth

failed = []
for rel_path in tqdm(paths, desc = 'Downloading'):
    local_path = os.path.join(OUT_DIR, rel_path.replace('/', os.sep))
    os.makedirs(os.path.dirname(local_path), exist_ok = True)

    if os.path.exists(local_path) and os.path.getsize(local_path) > 0:
        continue

    url = BASE_URL + rel_path
    try:
        r = session.get(url, timeout = 30)
        if r.status_code == 200:
            with open(local_path, 'wb') as f:
                f.write(r.content)
        else:
            failed.append((rel_path, r.status_code))
    except Exception as e:
        failed.append((rel_path, str(e)))

print(f'Done. Failed: {len(failed)}')
if failed:
    pd.DataFrame(failed, columns = ['path', 'error']).to_csv('failed_downloads.csv', index = False)

PhysioNet password:  ········


Downloading: 100%|██████████| 6137/6137 [34:04<00:00,  3.00it/s]  

Done. Failed: 6137


In [69]:
import requests
from requests.auth import HTTPBasicAuth
import getpass

username = "omerresearches"
password = getpass.getpass("PhysioNet password: ")

url = "https://physionet.org/files/mimic-cxr-jpg/2.1.0/files/p10/p10001884/s57156853/9fd47edd-07087209-b901811e-3e9e5f50-f382f611.jpg"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
}

r = requests.get(url, auth=HTTPBasicAuth(username, password), headers=headers)
print("Status:", r.status_code)
print("Headers:", r.headers.get('Content-Type'))
print("First 200 bytes:", r.content[:200])

PhysioNet password:  ········


Status: 403
Headers: text/html; charset=utf-8
First 200 bytes: b'<!DOCTYPE html>\n\n<html lang="en">\n  <head>\n    <meta charset="UTF-8">\n    <title>\n403: Forbidden Access\n</title>\n    \n    \n<link rel="stylesheet" type="text/css" href="/static/bootstrap/css/bootstrap.'


In [70]:
with open(r"D:\omer files\projects\NMIMS\cxr_project\test.jpg", "rb") as f:
    print(f.read(200))

b'<!DOCTYPE html>\n\n<html lang="en">\n  <head>\n    <meta charset="UTF-8">\n    <title>\n403: Forbidden Access\n</title>\n    \n    \n<link rel="stylesheet" type="text/css" href="/static/bootstrap/css/bootstrap.'


In [71]:
import pandas as pd

df = pd.read_csv('download_paths.csv')
base = "https://physionet.org/files/mimic-cxr-jpg/2.1.0/"
with open('urls.txt', 'w') as f:
    for p in df['jpg_path']:
        f.write(base + p + '\n')
print(f"Wrote {len(df)} URLs to urls.txt")

Wrote 6137 URLs to urls.txt
